# Day 12: Hybrid search

Day 11 ended with a failure: **"What does Section 103 say?"** and vector search never finds Section 103.

Today: why **vector search alone is not enough**, what **BM25** (keyword search) does better, and how **hybrid search** joins the two.

```
question ─┬─► vector search  (meaning)  ─► ranked list ─┐
          │                                             ├─► RRF ─► one list
          └─► BM25           (keywords) ─► ranked list ─┘
```

| § | Topic |
|---|---|
| 0 | Setup |
| 1 | Where vector search fails |
| 2 | BM25: keyword search with a score |
| 3 | Where BM25 fails |
| 4 | Hybrid search with Reciprocal Rank Fusion |
| 5 | Hybrid search on Kanoon.ai |
| 6 | Recap |

Needs `OPENAI_API_KEY` in `kanoon-ai/data_ingestion/.env` (the same key as Day 11).

## 0. Setup

In [1]:
import json, re
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from rank_bm25 import BM25Okapi

KANOON = Path("../../kanoon-ai/data_ingestion")
load_dotenv(KANOON / ".env")                     # the Kanoon.ai project's keys
client = OpenAI()

EMBED_MODEL = "text-embedding-3-small"           # the model of the saved BNS vectors (Day 10)

def embed(texts):
    data = client.embeddings.create(model=EMBED_MODEL, input=texts).data
    return np.array([d.embedding for d in data])

## 1. Where vector search fails

**Theory:** an embedding stores the **meaning** of a text (Day 10). Great for "killing someone" → murder. But some words have no "meaning" to find: **IDs, codes, numbers**. `INV-2023-091` and `INV-2023-019` mean the same thing to a model: "an invoice number". To the user they are two different invoices.

A small shop's data: 3 invoices, 2 error codes, 2 cars, 1 help page.

In [2]:
DOCS = [
    "Invoice INV-2023-091: 12 laptops sold to Sharma Traders, Bengaluru. Total ₹9,40,000. Status: paid.",
    "Invoice INV-2023-019: 3 printers sold to Sharma Traders, Bengaluru. Total ₹54,000. Status: pending.",
    "Invoice INV-2023-910: 40 office chairs sold to Gupta & Sons, Pune. Total ₹2,20,000. Status: paid.",
    "Code E1042: payment gateway timeout. Wait 30 seconds and retry the payment.",
    "Code E1024: card declined by the bank. Ask the customer to use another card.",
    "Budget SUVs with 5 seats and a big boot: Tata Punch and Maruti Brezza, both under ₹10 lakh.",
    "Luxury sedans: the BMW 5 Series and Mercedes E-Class start above ₹70 lakh.",
    "To reset your password, click 'Forgot password' on the login page.",
]
doc_vectors = embed(DOCS)

def vector_rank(query):                          # all doc numbers, best first
    scores = doc_vectors @ embed([query])[0]     # length-1 vectors, so dot = cosine (Day 10)
    return list(np.argsort(-scores)), scores

def show(query, rank_fn, k=3):
    ranking, scores = rank_fn(query)
    return pd.DataFrame([{"rank": r + 1, "doc": DOCS[i][:55], "score": round(float(scores[i]), 3)}
                         for r, i in enumerate(ranking[:k])], columns=["rank", "doc", "score"])

In [3]:
show("INV-2023-091", vector_rank)

,rank,doc,score
0,1,Invoice INV-2023-019: 3 printers sold to Sharm...,0.420
1,2,Invoice INV-2023-091: 12 laptops sold to Sharm...,0.414
2,3,Invoice INV-2023-910: 40 office chairs sold to...,0.371


We asked for **091** and got **019** first. 091 is only 2nd, and the scores are almost the same (0.420 vs 0.414): the model **can't tell them apart**.

A user (or the LLM) takes the 1st hit: the wrong invoice, **₹54,000 instead of ₹9,40,000**, and nothing looks wrong.

In [4]:
show("error E1042", vector_rank)

,rank,doc,score
0,1,Code E1024: card declined by the bank. Ask the...,0.458
1,2,Code E1042: payment gateway timeout. Wait 30 s...,0.448
2,3,Invoice INV-2023-019: 3 printers sold to Sharm...,0.235


Same with error codes. We asked for **E1042** (timeout) and got **E1024** (card declined) first: 0.458 vs 0.448, a coin toss. The support bot tells the customer to use another card, for a timeout.

**Why?** Look at how close the IDs are to each other for the embedding model:

In [5]:
ids = ["INV-2023-091", "INV-2023-019", "E1042", "E1024", "laptops"]
v = embed(ids)
pd.DataFrame(np.round(v @ v.T, 2), index=ids, columns=ids)

,INV-2023-091,INV-2023-019,E1042,E1024,laptops
INV-2023-091,1.00,0.93,0.43,0.36,0.19
INV-2023-019,0.93,1.00,0.42,0.36,0.18
E1042,0.43,0.42,1.00,0.68,0.21
E1024,0.36,0.36,0.68,1.00,0.19
laptops,0.19,0.18,0.21,0.19,1.00


To the model, `INV-2023-091` and `INV-2023-019` are **0.93 alike**: same digits, different order, almost the same vector. `E1042` vs `E1024`: 0.68. The embedding sees "an invoice number", not **which** one.

| Vector search is | Vector search is not |
|---|---|
| good at **meaning**: other words, other phrasing, other languages | good at **exact** tokens: IDs, codes, section numbers, names |
| "close enough" | "exactly this one" |

And it never says "no match": it always returns the *nearest* thing, even when it is wrong. The user sees an answer about the **wrong invoice**.

## 2. BM25: keyword search with a score

**Theory:** BM25 is the classic **keyword** ranking of search engines (the default in Elasticsearch). No model, no embeddings: it splits text into words and counts.

A document scores high when:
1. **TF (term frequency):** the query word appears in it, more often = better (but with diminishing returns).
2. **IDF (inverse document frequency):** the word is **rare** across all documents. Rare words matter more. `inv-2023-091` is in 1 doc, `invoice` is in 3.
3. **Length:** a short doc with the word beats a long doc with the word once.

Step 1, split into words (tokens):

In [6]:
def tokenize(text):
    return re.findall(r"[\w-]+", text.lower())  # words and codes; keeps "INV-2023-091" in one piece

print(tokenize("Show me invoice INV-2023-091"))

['show', 'me', 'invoice', 'inv-2023-091']


The `-` stays inside a word, so a code is **one** token. With `\w+` it would be `inv`, `2023`, `091`, and every 2023 invoice would partly match.

Step 2, build the index and search. `rank_bm25` does the counting:

In [7]:
bm25 = BM25Okapi([tokenize(d) for d in DOCS])

def bm25_rank(query):
    scores = bm25.get_scores(tokenize(query))
    return [i for i in np.argsort(-scores) if scores[i] > 0], scores    # 0 = no shared word: not a hit

show("INV-2023-091", bm25_rank)

,rank,doc,score
0,1,Invoice INV-2023-091: 12 laptops sold to Sharm...,1.566


BM25 returns **only the right invoice**. The other docs don't contain the word `inv-2023-091`, so they score 0 and are not hits at all.

Two things vector search can't do: an **exact** match, and saying **"no match"** (an empty list).

The weight of each word (IDF):

In [8]:
{w: round(bm25.idf[w], 2) for w in tokenize("invoice INV-2023-091")}

{'invoice': 0.45, 'inv-2023-091': 1.61}

`invoice` is in 3 of 8 docs: weight 0.45. `inv-2023-091` is in 1: weight 1.61, more than 3× as much. **Rare words decide the ranking.** Words that are in most docs (`the`, `and`) get almost no weight.

In [9]:
show("error E1042", bm25_rank)

,rank,doc,score
0,1,Code E1042: payment gateway timeout. Wait 30 s...,1.726


E1042 is 1st. E1024 isn't even listed: it doesn't contain `e1042`.

| Benefit | Limitation |
|---|---|
| exact matches: IDs, codes, section numbers, names | no meaning: different words = no match (§3) |
| fast, no API calls, no cost, runs anywhere | needs a good tokenizer (how you split words decides what can match) |
| explainable: you can see which word matched | every language, spelling and typo is a different word |

## 3. Where BM25 fails

**Theory:** BM25 only matches **the same words**. The user says "cheap family car", the doc says "Budget SUVs with 5 seats". Same intent, zero shared words.

In [10]:
show("cheap family car", bm25_rank)

,rank,doc,score


**Nothing.** No doc contains `cheap`, `family` or `car`. The right answer (the SUV doc) shares zero words with the question. BM25 also misses typos (`invoce`), other forms (`car` vs `cars`) and other languages (Hinglish, §5.4).

In [11]:
show("cheap family car", vector_rank)

,rank,doc,score
0,1,Budget SUVs with 5 seats and a big boot: Tata ...,0.415
1,2,Luxury sedans: the BMW 5 Series and Mercedes E...,0.269
2,3,Invoice INV-2023-910: 40 office chairs sold to...,0.111


Vector search finds the SUV doc, 0.414, far ahead of the luxury sedans (0.269). It knows that *budget* ≈ *cheap* and *SUV with 5 seats* ≈ *family car*.

| | finds meaning | finds exact words |
|---|---|---|
| Vector search | ✅ | ❌ |
| BM25 | ❌ | ✅ |

Each one fails where the other is good. So: **use both.**

## 4. Hybrid search with Reciprocal Rank Fusion

**Theory:** run both searches, then **merge** the two lists into one.

Can we just add the scores? No. They are on **different scales**: cosine was 0.1 to 0.5 here, BM25 has **no fixed range** (1.57 for the invoice, 1.73 for E1042, and other numbers on another corpus). Add them, and whichever runs bigger wins.

**Reciprocal Rank Fusion (RRF)** ignores the scores and uses only the **rank**:

```
RRF score(doc) = 1 / (60 + rank in list 1)  +  1 / (60 + rank in list 2)
```

- 1st in a list → 1/61, 2nd → 1/62, …: being near the top counts most.
- A doc in **both** lists gets two shares, so it rises.
- `60` is the constant from the paper (Cormack et al., 2009). It stops the 1st place from counting much more than the 2nd.

In [12]:
RRF_K = 60

def rrf(rankings, k=RRF_K):                      # rankings = lists of doc ids, best first
    scores = {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, 1):
            scores[doc] = scores.get(doc, 0) + 1 / (k + rank)
    return sorted(scores, key=lambda d: -scores[d]), scores

def hybrid_rank(query, top=5):                   # top 5 from each search, then fuse
    return rrf([vector_rank(query)[0][:top], bm25_rank(query)[0][:top]])

The RRF score, step by step, for `INV-2023-091`:

In [13]:
q = "INV-2023-091"
v, b = vector_rank(q)[0][:5], bm25_rank(q)[0][:5]
ranking, scores = hybrid_rank(q)
pd.DataFrame([{"doc": DOCS[d][:40], "vector rank": v.index(d) + 1 if d in v else "-",
               "bm25 rank": b.index(d) + 1 if d in b else "-", "RRF score": round(scores[d], 4)}
              for d in ranking])

,doc,vector rank,bm25 rank,RRF score
0,Invoice INV-2023-091: 12 laptops sold to,2,1,0.0325
1,Invoice INV-2023-019: 3 printers sold to,1,-,0.0164
2,Invoice INV-2023-910: 40 office chairs s,3,-,0.0159
3,Code E1024: card declined by the bank. A,4,-,0.0156
4,Code E1042: payment gateway timeout. Wai,5,-,0.0154


- **091:** 2nd in vector, 1st in BM25 → two shares, **0.0325**.
- **019:** 1st in vector only → one share, 0.0164.

091 wins by a lot. **Being in both lists counts more than being 1st in one.**

Now all three on every query: which doc comes **1st**?

In [14]:
rows = []
for q in ["INV-2023-091", "error E1042", "cheap family car", "how do I change my password"]:
    top = lambda rank_fn: DOCS[rank_fn(q)[0][0]][:32] if rank_fn(q)[0] else "(nothing)"
    rows.append({"query": q, "vector": top(vector_rank), "bm25": top(bm25_rank), "hybrid": top(hybrid_rank)})
pd.DataFrame(rows)

,query,vector,bm25,hybrid
0,INV-2023-091,Invoice INV-2023-019: 3 printers,Invoice INV-2023-091: 12 laptops,Invoice INV-2023-091: 12 laptops
1,error E1042,Code E1024: card declined by the,Code E1042: payment gateway time,Code E1042: payment gateway time
2,cheap family car,Budget SUVs with 5 seats and a b,(nothing),Budget SUVs with 5 seats and a b
3,how do I change my password,"To reset your password, click 'F","To reset your password, click 'F","To reset your password, click 'F"


Hybrid is right on all four, **without knowing in advance which kind of question it is**:
- **IDs and codes:** BM25 knew, vector didn't → hybrid right.
- **cheap family car:** BM25 found nothing, vector knew → hybrid right (an empty list adds nothing).
- **password:** both agree → hybrid agrees.

| Benefit | Limitation |
|---|---|
| exact words **and** meaning, in one list | two searches, two indexes to keep in sync |
| no tuning: no weights, scores on any scale | throws the scores away: "a bit better" and "much better" are just ranks |
| one more search costs almost nothing (BM25 is local) | when one list is pure noise, it can push a good doc down (§5) |

The other way: **weighted fusion**, `α × vector score + (1 − α) × BM25 score`, after scaling both to 0–1. You can tune `α`, but you **have to**. RRF is the safe default.

## 5. Hybrid search on Kanoon.ai

The 198 BNS chunks and the vectors saved on Day 10, same as Day 11:

In [15]:
chunks = [json.loads(line) for line in (KANOON / "data/chunks/bns.jsonl").open()]
vectors = np.load(KANOON / "data/embeddings/bns_vectors.npy")
print(len(chunks), "chunks |", vectors.shape, "vectors")

def bns_vector(query, k=10):
    scores = vectors @ embed([query])[0]
    return [int(i) for i in np.argsort(-scores)[:k]]

198 chunks | (198, 1536) vectors


### 5.1 BM25 on the law, first try

In [16]:
bns_bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])

def bns_keyword(query, k=10, index=None):
    scores = (index or bns_bm25).get_scores(tokenize(query))
    return [int(i) for i in np.argsort(-scores)[:k] if scores[i] > 0]   # 0 = no shared word: not a hit

def ids(ranking, n=5):
    return [chunks[i]["chunk_id"] for i in ranking[:n]]

q = "Section 103"
print("vector:", ids(bns_vector(q)))
print("bm25  :", ids(bns_keyword(q)))

vector: ['bns::5', 'bns::6', 'bns::3', 'bns::197', 'bns::110']
bm25  : ['bns::114', 'bns::129', 'bns::125', 'bns::61', 'bns::108']


Vector search: no `bns::61`, same as Day 11. BM25 finds chunks with `103` in them, but the real Section 103 (`bns::61`) is only **4th**. What is above it?

In [17]:
for i in bns_keyword(q)[:3]:
    text = chunks[i]["text"]
    at = text.find("103")
    print(chunks[i]["chunk_id"], "…", text[at - 70:at + 30].replace("\n", " "), "…")

bns::114 … dia, would be punishable under any of the following sections, namely, 103, 105, 307, sub-sections (2 …
bns::129 … dia, would be punishable under any of the following sections, namely, 103, 105, 307, sub-sections (2 …
bns::125 … dia, would be punishable under any of the following sections, namely, 103, 105, 307, sub-sections (2 …


Three chunks that **mention** 103 in a list of other sections. BM25 counts words; it doesn't know which `103` is the heading of the section and which is just a mention.

### 5.2 Index the section numbers

**Theory:** what you put in the BM25 index decides what can match. Each BNS section starts with its number in bold, `**103.**`. Pull those numbers out and add `Section 103` to the chunk's **BM25 text**. The chunk the user reads stays the same.

(Same idea as the breadcrumb on Day 09: add the words people search with.)

In [18]:
SECTION_START = re.compile(r"\*\*(\d+)\.\*\*")    # "**103.**" = Section 103 starts here

def bm25_text(chunk):
    sections = " ".join(f"Section {n}" for n in SECTION_START.findall(chunk["text"]))
    return sections + "\n" + chunk["text"]

print(bm25_text(chunks[61])[:60], "…")
bns_bm25_sections = BM25Okapi([tokenize(bm25_text(c)) for c in chunks])
print("bm25  :", ids(bns_keyword(q, index=bns_bm25_sections)))

Section 102 Section 103 Section 104 Section 105
in the heat  …
bm25  : ['bns::61', 'bns::114', 'bns::129', 'bns::125', 'bns::108']


`bns::61` is now **1st**. One line of preprocessing fixed BM25, and the embeddings didn't change.

### 5.3 Vector vs BM25 vs hybrid

Questions with the chunk that holds the answer (from Day 11):

In [19]:
RIGHT = {
    "What does Section 103 say?": ["bns::61"],
    "What is Section 356?": ["bns::191"],
    "Section 303 theft": ["bns::148"],
    "punishment for murder": ["bns::61"],
    "Is killing someone a crime?": ["bns::57", "bns::61"],
    "someone posted lies about me on facebook": ["bns::191", "bns::192"],
}

def rank_of_right(question, ranking):
    ranks = [n + 1 for n, cid in enumerate(ids(ranking, n=len(ranking))) if cid in RIGHT[question]]
    return ranks[0] if ranks else "-"

def bns_hybrid(query, k=10):
    return rrf([bns_vector(query, k), bns_keyword(query, k, bns_bm25_sections)])[0]

rows = []
for q in RIGHT:
    rows.append({"question": q, "vector": rank_of_right(q, bns_vector(q)),
                 "bm25": rank_of_right(q, bns_keyword(q, index=bns_bm25_sections)),
                 "hybrid": rank_of_right(q, bns_hybrid(q))})
scoreboard = pd.DataFrame(rows)
scoreboard

,question,vector,bm25,hybrid
0,What does Section 103 say?,-,1,2
1,What is Section 356?,-,1,2
2,Section 303 theft,7,1,3
3,punishment for murder,1,2,1
4,Is killing someone a crime?,2,-,4
5,someone posted lies about me on facebook,5,-,9


In [20]:
in_top5 = lambda col: sum(r != "-" and r <= 5 for r in scoreboard[col])
print({col: f"{in_top5(col)} of {len(scoreboard)} in the top 5" for col in ["vector", "bm25", "hybrid"]})

{'vector': '3 of 6 in the top 5', 'bm25': '4 of 6 in the top 5', 'hybrid': '5 of 6 in the top 5'}


- **Section numbers:** vector search never finds them, BM25 puts them 1st, hybrid 2nd or 3rd. (The 1st for "Section 103" is `bns::5`, page 4, a long list of section numbers: vector's top hit, so it keeps a share.)
- **punishment for murder:** both find it, hybrid 1st.
- **killing someone:** vector 2nd, hybrid 4th: BM25 brought other murder chunks (`bns::64`, `bns::66`), not wrong, just not these two.
- **facebook:** the BNS never says `facebook`, so BM25 matched small words (`about`, `on`, `posted`, `lies` in another sense) in unrelated chunks. That noise pushed defamation from 5th to **9th**.

**Hybrid isn't the best on every question, but it's rarely bad:** 5 of 6 in the top 5, against 3 (vector) and 4 (BM25). Getting the best chunk to **1st** inside that top 10 is the **reranker's** job (Day 13).

### 5.4 What hybrid can't fix

In [21]:
q = "hatya ki saza kya hai"
print("vector:", ids(bns_vector(q)))
print("bm25  :", ids(bns_keyword(q, index=bns_bm25_sections)), "← no shared word")
print("hybrid:", ids(bns_hybrid(q)))

vector: ['bns::159', 'bns::73', 'bns::151', 'bns::79', 'bns::25']
bm25  : [] ← no shared word


hybrid: ['bns::159', 'bns::73', 'bns::151', 'bns::79', 'bns::25']


`hatya` is not in the English BNS, so BM25 has nothing, and hybrid = vector search, which fails (Day 11). **Hybrid doesn't fix the vocabulary gap.** Kanoon.ai first rewrites the question (Day 11), then runs hybrid search. That's homework 4.

## 6. Recap

| Step | What | Function |
|---|---|---|
| Vector search | meaning: other words, other phrasing | `vector_rank()`, `bns_vector()` |
| Tokenize | text → words; decides what BM25 can match | `tokenize()` |
| BM25 | exact words, rare words count more | `BM25Okapi`, `bm25_rank()`, `bns_keyword()` |
| Index the right words | add `Section 103` to the BM25 text | `bm25_text()` |
| Fuse | rank-only merge, `1 / (60 + rank)` | `rrf()` |
| Hybrid | both searches → RRF | `hybrid_rank()`, `bns_hybrid()` |

**When to use what**

| Your users search for… | Use |
|---|---|
| ideas, questions in their own words | vector |
| IDs, codes, SKUs, names, section numbers | BM25 |
| both (almost every real app) | **hybrid** |

In production the database does this for you: Qdrant runs a sparse (BM25-style) search and a vector search as two `prefetch` queries and fuses them with `RrfQuery` (link below). Elasticsearch uses BM25 as its default ranking.

**Homework**
1. Add 3 docs to `DOCS` with codes that look alike (`SKU-7781`, `SKU-7718`…). Does vector search mix them up? Does hybrid?
2. In `rrf()`, try `k=1` and `k=200`. How does the `INV-2023-091` table change?
3. Write 5 BNS questions with a section number and 5 without. Score vector, BM25 and hybrid like §5.3.
4. Hybrid with a **rewritten** query (Day 11's `rewrite()`) on "hatya ki saza kya hai". Does Section 103 come back?

Sources (checked 2026-10-31): BM25 library `rank_bm25` https://github.com/dorianbrown/rank_bm25 · RRF paper (Cormack, Clarke, Büttcher, SIGIR 2009) http://cormack.uwaterloo.ca/cormack/cormacksigir09-rrf.pdf · BEIR benchmark, BM25 is "a robust baseline" https://arxiv.org/abs/2104.08663 · Qdrant hybrid queries https://qdrant.tech/documentation/search/hybrid-queries/ · Elasticsearch, BM25 is the default similarity https://www.elastic.co/docs/reference/elasticsearch/index-settings/similarity